# Efecto de $S(\alpha,\beta)$ en el WCD: análisis por medio detector

Barrido de neutrones monoenergéticos de 1 meV a 1 keV (16 energías, 10 000 neutrones por punto) con dos físicas:
`QGSP_BERT_HP` (con $S(\alpha,\beta)$) y `QGSP_BERT_HP_NoThermal` (gas libre). Geometría de la Campaña 2
(agua de 48 × 133 cm, Tyvek 0.12 mm, acero 0.5 mm, PMT a 86.5 cm del fondo), incidencia vertical desde z = 200 cm.

Medios disponibles: agua pura y agua + 2.5 % de NaCl. Las campañas con 5 % y 10 % se añaden ejecutando la celda de
extracción con su carpeta de datos.

**Programas** (en esta carpeta):

| Programa | Función |
|---|---|
| `analizar_medio.sh <MEDIO> <carpeta>` | ejecuta toda la cadena para un medio y escribe en `salida/<MEDIO>/` |
| `balance_destinos.sh`, `capturas_detalle.sh` | destino de cada neutrón; profundidad, tiempo y energía de captura |
| `em_detalle.sh`, `em_cadena.sh` | γ de captura (con el núcleo de captura), carga, electrones y luz Cherenkov |
| `figuras_*.py` | figuras y resúmenes por medio |
| `comparar_medios.py` | comparación entre medios (`salida/comparacion/`) |

In [ ]:
from pathlib import Path
import subprocess
import pandas as pd
from IPython.display import Image, display

BASE = Path.cwd()
DATOS = BASE.parent / "Resultados-simulaciones"
CARPETAS = {
    "AguaPura": DATOS / "Prueba-agua-pura-1meV-1keV-dos-libreria",
    "Agua25NaCl": DATOS / "Prueba-agua-25NaCl-1meV-1keV-dos-libreria",
    "Agua5NaCl": DATOS / "Prueba-agua-5NaCl-1meV-1keV-dos-libreria",
    "Agua10NaCl": DATOS / "Prueba-agua-10NaCl-1meV-1keV-dos-libreria",
}
NOMBRE = {"AguaPura": "agua pura", "Agua25NaCl": "2.5 % NaCl", "Agua5NaCl": "5 % NaCl", "Agua10NaCl": "10 % NaCl"}
SAB, LIBRE = "QGSP_BERT_HP", "QGSP_BERT_HP_NoThermal"
pd.set_option("display.width", 200)
{m: c.exists() for m, c in CARPETAS.items()}

## 1. Extracción

Ejecuta la cadena completa (unos minutos por medio) solo para los medios cuyos datos existen y que aún no tienen resultados. Poner `REHACER = True` para repetirla.

In [ ]:
REHACER = False
for m, carpeta in CARPETAS.items():
    hecho = (BASE / "salida" / m / "resumen_em2.tsv").exists()
    if carpeta.exists() and (REHACER or not hecho):
        print(f"Analizando {m}...")
        subprocess.run(["./analizar_medio.sh", m, str(carpeta)], check=True)
MEDIOS = [m for m in CARPETAS if (BASE / "salida" / m / "resumen_em.tsv").exists()]
MEDIOS

## 2. Captura en el agua y número de dispersiones

In [ ]:
def tabla(archivo, columnas):
    filas = []
    for m in MEDIOS:
        t = pd.read_csv(BASE / "salida" / m / archivo, sep="\t")
        t["medio"] = NOMBRE[m]
        filas.append(t)
    t = pd.concat(filas)
    return t.pivot_table(index="energia_eV", columns=["medio", "fisica"], values=columnas)

tabla("resumen_barrido.tsv", ["captura_agua_pct", "N_medio"]).round(2)

## 3. Energía de cruce del efecto de $S(\alpha,\beta)$

Interpolación logarítmica donde la diferencia con − sin cambia de signo. Entre 0.3 y 0.5 eV las diferencias son compatibles con cero, por lo que el cruce tiene una incertidumbre de varias décimas de eV.

In [ ]:
subprocess.run(["python3", "comparar_medios.py"], check=True, capture_output=True)
pd.read_csv(BASE / "salida" / "comparacion" / "cruces.tsv", sep="\t")

## 4. Eficiencia de detección y ganancia con el umbral de carga

In [ ]:
filas = []
for m in MEDIOS:
    e = pd.read_csv(BASE / "salida" / m / "em_eventos.tsv", sep="\t")
    for (E, f), g in e.groupby(["energia_eV", "fisica"]):
        filas.append({"medio": NOMBRE[m], "energia_eV": E, "fisica": f,
                      **{f"Q>={u}": 100 * (g.carga_pe >= u).mean() for u in (1, 3, 5)}})
ef = pd.DataFrame(filas)
ef[ef.fisica == SAB].pivot_table(index="energia_eV", columns="medio", values=["Q>=1", "Q>=3", "Q>=5"]).round(2)

In [ ]:
# Ganancia de cada medio con NaCl respecto al agua pura, con S(alpha,beta)
p = ef[ef.fisica == SAB].set_index(["energia_eV", "medio"])[["Q>=1", "Q>=3", "Q>=5"]]
ganancias = {NOMBRE[m]: (p.xs(NOMBRE[m], level="medio") / p.xs("agua pura", level="medio")).round(2)
             for m in MEDIOS if m != "AguaPura"}
pd.concat(ganancias, axis=1) if ganancias else "Solo hay agua pura"


## 5. Capturas por núcleo y respuesta por núcleo

In [ ]:
filas = []
for m in MEDIOS:
    e = pd.read_csv(BASE / "salida" / m / "em_eventos.tsv", sep="\t")
    c = e[(e.destino == "captura_agua") & (e.fisica == SAB)]
    g = c.groupby("nucleo").agg(capturas=("carga_pe", "size"), gammas_por_captura=("n_gammas", "mean"),
                                gammas_interactuan=("n_gammas_interactuan_agua", "mean"),
                                energia_agua_MeV=("energia_transferida_agua_MeV", "mean"),
                                P_senal_pct=("carga_pe", lambda q: 100 * (q >= 1).mean()),
                                P_Q3_pct=("carga_pe", lambda q: 100 * (q >= 3).mean()))
    g["fraccion_pct"] = 100 * g.capturas / g.capturas.sum()
    g.insert(0, "medio", NOMBRE[m])
    filas.append(g)
pd.concat(filas).round(2)

## 6. Cadena electromagnética (resumen por medio)

In [ ]:
pd.concat({NOMBRE[m]: pd.read_csv(BASE / "salida" / m / "resumen_em2.tsv", sep="\t").set_index("magnitud")["valor"]
           for m in MEDIOS}, axis=1)

## 7. Figuras comparativas

In [ ]:
display(Image(filename=str(BASE / "salida" / "comparacion" / "fig_comparacion_medios.png"), width=900))
if (BASE / "salida" / "comparacion" / "fig_captura_por_nucleo.png").exists():
    display(Image(filename=str(BASE / "salida" / "comparacion" / "fig_captura_por_nucleo.png"), width=600))

## 8. Figuras por medio

In [ ]:
for m in MEDIOS:
    print(NOMBRE[m])
    for f in ("fig_captura_N_vs_energia", "fig_eficiencia_deteccion", "fig_contencion_gamma", "fig_luz_carga"):
        display(Image(filename=str(BASE / "salida" / m / f"{f}.png"), width=800))